# Provisioning resources

## Prerequisites

Before starting, make sure you:

- Have completed the [Preparation](./00_preparation.ipynb) tutorial.
- Have a basic understanding of Python.

## Introduction

5G experiments run on physical or virtual infrastructure resources. To set up and configure these resources to host 5G workloads, the framework relies on **resource providers** and **provisioners**.

### Core Concepts

* **Metamodel:** Describes the abstract 5G network topology and network functions. Because metamodels are implementation-agnostic, you can easily share and run your 5G network definitions across different environments and protocol implementations.
* **Deployment Model:** Complements the abstract metamodel by supplying the technical parameters required for a specific physical or virtual deployment.
* **Provisioners:** Explicitly declare the infrastructure nodes that will host 5G functions. By listing and tagging machines with appropriate roles in the `nodes` section of the deployment model, the orchestrator library automatically schedules network functions onto compatible nodes.

The orchestrator offers a collection of built-in provisioners tailored for various resource providers, each with its own specific configuration parameters.

### Tutorial Scope

This tutorial focuses exclusively on **infrastructure provisioning** and does not deploy an active 5G network. Nevertheless, because provisioners always require a valid metamodel reference, you will instantiate and use an empty metamodel throughout this guide.

In [1]:
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel()

Provisioners are managed using instances of the `ProvisioningManagement` class, which automatically determine how to configure and provision resources based on the supplied deployment model.

## `null` Provisioner

The `null` provisioner handles pre-existing infrastructure. It indicates that no automated provisioning action is required for the declared nodes; instead, it assumes that the nodes are already active, running, and fully configured to host workloads orchestrated by the library.

### Manual Provisioning Flag

Because no automated provider operations take place, you must explicitly inform the library that a node is available by setting the `is_provisioned` attribute to `true`. When using automated provisioners, you do not need to manage this attribute manually, as the provisioner automatically populates it based on the provisioning status.

### SSH Access & Authentication Configurations

The library always communicates with target nodes over SSH. Below are common authentication patterns:

* **Default Configuration (`default.example.org`):** Accessed using default SSH parameters. By default, the username is `root`, and the client leverages the default keys and settings of the local user running the library.
* **Sudo User (`password.example.org`):** Connected via the `ubuntu` user using default SSH keys. Because administrative privileges require elevation, the configuration explicitly supplies the `sudo` password for the `ubuntu` user.
* **Custom SSH Key (`machine-key`):** Used when a node cannot be accessed via the local user's default SSH keys. A path to a specific private key is explicitly provided.

> **Important:** When using the `null` provisioner, if no hostname is provided, the the node name in the metamodel is sued as the machine's actual, resolvable hostname or IP address to establish SSH connections.

In [2]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
nodes:
  default.example.org: # node's actual hostname
    is_provisioned: true
    provisioner: null

  password.example.org: # node's actual hostname
    is_provisioned: true
    provisioner: null
    ssh_info:
      user: ubuntu
      sudo_password: password

  machine-key:
    is_provisioned: true
    ssh_info:
      hostname: key.example.org
      private_key_file: /home/user/.ssh/id_ed25519_custom
"""
)

Because the `null` provisioner performs no actual management tasks, no further orchestration steps are necessary as long as the node is marked as `is_provisioned: true`.

> **Note:** Users are responsible for ensuring that manually provisioned nodes meet all system requirements and are fully compliant with the orchestrator library.

## SLICES BI Provisioner

The SLICES BI provisioner is used to create ephemeral virtual machines on the [SLICES Basic Infrastructure Service (SLICES BI)](https://doc.slices-ri.eu/BasicServices/BasicInfrastructure/basicinfrastructure.html).

Using this provisioner requires the SLICES CLI to be installed on your local machine. Refer to the [Preparation tutorial](./preparation.ipynb) for installation instructions and a review of fundamental SLICES-RI concepts.

> **Important:** Ensure your public SSH keys are registered with SLICES before deploying your network.

### Configuring the Deployment Model

Having already defined the metamodel, you must now configure the deployment model to specify the target infrastructure for provisioning.

The `ProvisioningManagement` manager provisions the machines declared in the `nodes` section of the deployment model, automatically determining the correct provisioner for each node using the configurations defined in the `provisioners` section.

In this example, you will deploy three virtual machines (`node1`, `node2`, and `node3`) on the `be-gent1-bi-vm1` SLICES-BI infrastructure (you can list the available LICES-BI infrastructures with the `slices bi infra list --all`). These nodes will run within the `provision_tutorial` experiment under the `post5g-beta` project. The configuration instructs the provisioner to automatically create the experiment and assigns the nodes a 1-hour lifetime, after which they will be automatically destroyed. These parameters are defined in a custom provisioner named `my-provisioner` within the `provisioners` section of the deployment model.

Once the provisioner is defined, you declare the target nodes and link them to it:
* **`node1`**: Tagged with the `core` role.
* **`node2`**: Tagged with the `ran` role.
* **`node3`**: Tagged to support both `ran` and `ue` roles.

All three nodes are provisioned using `my-provisioner`.

In [3]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
provisioners:
  my-provisionner:
    type: bi
    params:
      infra: be-gent1-bi-vm1
      project: post5g-beta
      experiment: provision_tutorial
      create_experiment: true
      experiment_duration: 1h
      node_duration: 1h
nodes:
  node1:
    provisioner: my-provisionner
    roles:
    - core
  node2:
    provisioner: my-provisionner
    roles:
    - ran
  node3:
    provisioner: my-provisionner
    roles:
    - ran
    - ue
"""
)


To provision the nodes, instantiate a `ProvisioningManagement` manager using the previously defined metamodel and deployment model, then invoke its `provision()` method.

In [4]:
from orchestrator.deployment import ProvisioningManagement
prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision(quiet=False)

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node1' provisioning] ***
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for '

Your three virtual machines are now successfully deployed on SLICES BI and configured for the scenario. You can verify the deployment by listing the experiment resources using the SLICES CLI (be sure to adjust the parameters to match your specific setup):

In [5]:
!SLICES_BI_SITE_ID=be-gent1-bi-vm1 slices bi list-resources --experiment provision_tutorial

                                Resources in experiment exp_expauth.ilabt.imec.be_01m5vtmn48fb580se6tz0a4w02                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ ID                                          ┃ Name  ┃ Status ┃ Created At           ┃ Expires At           ┃ Public IPv4 ┃ Private IPv4  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ r_be-gent1-bi-vm1_01m5vtmqd5fk8thsfmp0gp8r… │ node2 │ up     │ 2026-10-26 10:03 UTC │ 2026-10-26 11:03 UTC │             │ 10.10.214.236 │
│ r_be-gent1-bi-vm1_01m5vtmqdbetjtnm86ssvpdn… │ node1 │ up     │ 2026-10-26 10:03 UTC │ 2026-10-26 11:03 UTC │             │ 10.10.213.12  │
│ r_be-gent1-bi-vm1_01m5vtmr0dfdc92f48af3mfg… │ node3 │ up     │ 2026-10-26 10:03 UTC │ 2026-10-26 11:03 UTC │             │ 10.10.215.227 │
└────────────

## Saving the Metamodel and Deployment Model

Each manager accepts a metamodel and a deployment model as inputs and outputs an updated deployment model reflecting the results of its execution. This updated model can then be passed downstream to the next manager in the pipeline.

You can retrieve the updated metamodel and deployment model from a manager through several convenience methods. For instance, the `output()` method directly returns both models. The updated deployment model contains all technical credentials and parameters required to access the newly created virtual machines.

To persist the current infrastructure state for subsequent orchestration steps, you can save both models to file:

In [6]:
metamodel, deployment_model = prov.output()
from pathlib import Path
DIR="state/provision"
Path(DIR).mkdir(parents=True, exist_ok=True)

metamodel.to_yaml_file(f"{DIR}/metamodel.prov.yaml", exclude_defaults=False)
deployment_model.to_yaml_file(f"{DIR}/deployment_model.prov.yaml", exclude_defaults=False)

## Unprovisioning Nodes

Provisioners can also be used to unprovision the nodes they have previously deployed. 

You can choose to target specific nodes for unprovisioning or clean up all nodes declared within the deployment model at once.

The exact effect of unprovisioning depends on the underlying provisioner (for instance, the SLICES BI provisioner terminates and destroys the corresponding virtual machine). However, regardless of the provider, the node entry remains present in the deployment model—allowing it to be re-provisioned later—while its `is_provisioned` attribute is reset to `false`.

In [7]:
await prov.unprovision_nodes(nodes=["node1", "node2"])  # Unprovision some nodes
!SLICES_BI_SITE_ID=be-gent1-bi-vm1 slices bi list-resources --experiment provision_tutorial


await prov.unprovision()                                # Unprovision all nodes
!SLICES_BI_SITE_ID=be-gent1-bi-vm1 slices bi list-resources --experiment provision_tutorial

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/unprovision : Select project 'post5g-beta' for 'node2' unprovisioning] ***

TASK [provisioning/bi/unprovision : Select project 'post5g-beta' for 'node1' unprovisioning] ***
changed: [localhost]
changed: [localhost]

TASK [provisioning/bi/unprovision : Destroy 'node1' on infrastructure 'be-gent1-bi-vm1' using experiment 'provision_tutorial'] ***

TASK [provisioning/bi/unprovision : Destroy 'node2' on infrastructure '